In [4]:
from pathlib import Path

from helpers._helpers import run_experiment_ablation, run_sweeps_for_process_list
import numpy as np

from olbootstrap.online_bootstrap._block_bootstrap import (
    BlockBootstrap,
)
from olbootstrap.online_bootstrap._online_ar_bootstrap import OnlineARBootstrap
from olbootstrap.online_bootstrap._online_gaussian_bootstrap import (
    OnlineGaussianMixtureAsympCSSmoothedBootstrap,
)
from olbootstrap.study._study import (
    UniformCoverageStudyWithTesting,
)
from olbootstrap.synthetic_time_series._ar1 import AR1Process

In [5]:
seed_master = 1234
rng = np.random.default_rng(seed_master)

base_proc = AR1Process(
    mean=0.0,
    phi=0.3,
    noise_std=1.0,
    seasonal_amplitude=0.4,
    seasonal_period=400,
    seasonal_phase=0.0,
    rng=rng,
)

base_exp_kwargs = {
    "B": 400,
    "record_every": 1,
    "smoothing_method": "ewma",
    "method_label": "ARmmult",
    "bootstrap_cls": OnlineARBootstrap,
    "use_variance_smoothing": False,
}

res_dict = UniformCoverageStudyWithTesting.run_power_sweep(
    base_process_template=base_proc,
    etas=[2 / 20, 2 / 50, 2 / 100],
    phis=[0.3, 0.6],
    trend_slopes=[0.0001, 0.0005, 0.001],
    sample_size=3500,
    n_series=150,
    burn_in=500,
    var_warmup=400,
    alpha=0.1,
    base_exp_kwargs=base_exp_kwargs,
    outdir=Path("..") / "experiments" / "power_ewma_alpha-0p1",
    seed=seed_master,
    parallel=True,
    n_jobs=-1,
    transform="student",
    transform_power=1 / 3,
    rho_power=-1 / 3,
    save=True,
)
print("saved", len(res_dict), "runs")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed: 13.7min
[Parallel(n_jobs=-1)]: Done   5 out of  18 | elapsed: 13.8min remaining: 35.9min
[Parallel(n_jobs=-1)]: Done   7 out of  18 | elapsed: 13.8min remaining: 21.7min
[Parallel(n_jobs=-1)]: Done   9 out of  18 | elapsed: 26.7min remaining: 26.7min
[Parallel(n_jobs=-1)]: Done  11 out of  18 | elapsed: 26.7min remaining: 17.0min
[Parallel(n_jobs=-1)]: Done  13 out of  18 | elapsed: 26.8min remaining: 10.3min
[Parallel(n_jobs=-1)]: Done  15 out of  18 | elapsed: 26.9min remaining:  5.4min


saved 18 runs


[Parallel(n_jobs=-1)]: Done  18 out of  18 | elapsed: 36.6min finished


In [8]:
exp_kwargs_overrides = [
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmmult",
        "transform": "student",
    },
    {
        "bootstrap_cls": OnlineGaussianMixtureAsympCSSmoothedBootstrap,
        "method_label": "GaussMix",
        "transform": "student",
    },
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmrho0",
        "rho_power": 0.0,
        "transform": "gauss",
    },
    {
        "bootstrap_cls": BlockBootstrap,
        "method_label": "Block",
        "transform": "student",
    },
]

In [9]:
outdir = Path("..") / "experiments"
outdir.mkdir(parents=True, exist_ok=True)

seed_master = 1234
phis = [0.3, 0.6]

sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
var_warmup = 400

# panel_names = ["Trend + seasonality"]
panel_names = ["Stationary", "Trend + seasonality", "Trend + shocks"]


def make_panel_process(panel_name: str, phi: float, rng: np.random.Generator):
    if panel_name == "Stationary":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.0,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + seasonality":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.4,
            seasonal_period=400,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + shocks":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
            shock_type="permanent",
            jump_prob=0.005,
            jump_scale=2.0,
        )
    raise ValueError(f"Unknown panel: {panel_name}")


setups = [
    # ("ewma", 0.1),
    # ("ewma", 0.2),
    # ("brown", 0.1),
    ("brown", 0.2),
]

for setup_idx, (smooth_method, alpha) in enumerate(setups):
    base_exp_kwargs = {
        "B": 400,
        "record_every": 1,
        "smoothing_method": smooth_method,
    }

    setup_dir = outdir / f"smooth-{smooth_method}_alpha-{str(alpha).replace('.', 'p')}"
    setup_dir.mkdir(parents=True, exist_ok=True)

    for panel_idx, panel_name in enumerate(panel_names):
        processes = []
        labels = []
        for phi_idx, phi in enumerate(phis):
            rng = np.random.default_rng(
                seed_master + 10_000 * setup_idx + 1_000 * panel_idx + 10 * phi_idx
            )
            proc = make_panel_process(panel_name, phi, rng)
            processes.append(proc)
            labels.append(rf"phi={phi:g}")

        _ = run_sweeps_for_process_list(
            processes,
            labels=labels,
            outdir=setup_dir / panel_name,
            seed=seed_master + 100 * setup_idx + panel_idx,
            sample_size=sample_size,
            smoothing_grid=smoothing_grid,
            n_series=n_series,
            burn_in=burn_in,
            alpha=alpha,
            var_warmup=var_warmup,
            base_exp_kwargs=base_exp_kwargs,
            exp_kwargs_overrides=exp_kwargs_overrides,
            progress=True,
            parallel=True,
            n_jobs=-1,
        )

print("Done. Results saved under:", outdir.resolve())

[run_sweeps_for_process_list] Running sweeps for phi=0.3 -> ../experiments/smooth-brown_alpha-0p2/Stationary/phi=0.3


combo runs:   0%|          | 0/4 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  5.6min remaining:  8.4min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  5.6min remaining:  3.7min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.8min finished
combo runs:  25%|██▌       | 1/4 [05:45<17:16, 345.54s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   34.6s remaining:   51.8s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   34.7s remaining:   23.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   35.1s finished
combo runs:  50%|█████     | 2/4 [06:20<05:25, 162.92s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   43.9s remaining:  1.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   44.2

[run_sweeps_for_process_list] Running sweeps for phi=0.6 -> ../experiments/smooth-brown_alpha-0p2/Stationary/phi=0.6


combo runs:   0%|          | 0/4 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  5.4min remaining:  8.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  5.5min remaining:  3.6min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.6min finished
combo runs:  25%|██▌       | 1/4 [05:36<16:48, 336.16s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   33.9s remaining:   50.8s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   33.9s remaining:   22.6s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   35.0s finished
combo runs:  50%|█████     | 2/4 [06:11<05:17, 158.99s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   42.8s remaining:  1.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   42.8

[run_sweeps_for_process_list] Running sweeps for phi=0.3 -> ../experiments/smooth-brown_alpha-0p2/Trend + seasonality/phi=0.3


...

[run_sweeps_for_process_list] Running sweeps for phi=0.3 -> ../experiments/smooth-brown_alpha-0p2/Trend + shocks/phi=0.3


combo runs:   0%|          | 0/4 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  5.2min remaining:  7.9min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  5.3min remaining:  3.5min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.4min finished
combo runs:  25%|██▌       | 1/4 [05:25<16:16, 325.52s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   33.6s remaining:   50.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   33.6s remaining:   22.4s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   34.0s finished
combo runs:  50%|█████     | 2/4 [05:59<05:08, 154.02s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   42.5s remaining:  1.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   42.6

[run_sweeps_for_process_list] Running sweeps for phi=0.6 -> ../experiments/smooth-brown_alpha-0p2/Trend + shocks/phi=0.6


combo runs:   0%|          | 0/4 [00:00<?, ?it/s][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:  5.2min remaining:  7.9min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  5.3min remaining:  3.5min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  5.4min finished
combo runs:  25%|██▌       | 1/4 [05:25<16:15, 325.16s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   33.7s remaining:   50.6s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   33.8s remaining:   22.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   34.0s finished
combo runs:  50%|█████     | 2/4 [05:59<05:07, 153.89s/it][Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   42.7s remaining:  1.1min
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   42.8

Done. Results saved under: /Users/toby/bootstrap/experiments


In [ ]:
phi = 0.5
ar1_base = AR1Process(
    mean=0.0,
    phi=phi,
    noise_std=1.0,
    trend_slope=0.0,
    seasonal_amplitude=0.0,
    seasonal_period=None,
    seasonal_phase=0.0,
    rng=np.random.default_rng(0),
)

sample_size = 3500
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 150
burn_in = 500
alpha = 0.1
var_warmup_fixed = 400
seed_master = 1234

base_exp_kwargs = {
    "B": 200,
    "record_every": 1,
    "smoothing_method": "ewma",
}

In [ ]:
B_values = [20, 50, 100, 200]
res_B = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="B",
    param_values=B_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_B",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
vw_values = [20, 100, 200, 400]
res_vw = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="var_warmup",
    param_values=vw_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_vw",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
tp_values = [1 / 4, 1 / 3, 1 / 2, 1.0]
res_tp = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="transform_power",
    param_values=tp_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_tp",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    rho_power=-1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
rho_values = [-1 / 2, -1 / 3, -1 / 4, 0.0]
res_rho = run_experiment_ablation(
    base_process_template=ar1_base,
    param_name="rho_power",
    param_values=rho_values,
    outdir=Path("..") / "experiments" / "AR1_phi0p5_ablate_rho",
    sample_size=sample_size,
    smoothing_grid=smoothing_grid,
    n_series=n_series,
    burn_in=burn_in,
    alpha=alpha,
    var_warmup=var_warmup_fixed,
    base_exp_kwargs=base_exp_kwargs,
    transform="student",
    transform_power=1.0 / 3.0,
    seed_master=seed_master,
)

In [ ]:
exp_kwargs_overrides_ablation = [
    {
        "bootstrap_cls": OnlineARBootstrap,
        "method_label": "ARmmult",
        "transform": "student",
    }
]

smooth_method = "ewma"
alpha = 0.1

sample_size = 5000
smoothing_grid = [2 / 10, 2 / 20, 2 / 50, 2 / 100, 2 / 250]
n_series = 250
burn_in = 500
var_warmup = 400

base_exp_kwargs_ablation = {
    "B": 200,
    "record_every": 1,
    "smoothing_method": smooth_method,
}

phis = [0.2, 0.4, 0.6, 0.8]
panel_names_ablation = ["Stationary", "Trend + shocks"]

outdir = Path("..") / "experiments_test"
setup_dir = outdir / f"smooth-{smooth_method}_alpha-{str(alpha).replace('.', 'p')}"
abl_dir = setup_dir / "ablations_long"
abl_dir.mkdir(parents=True, exist_ok=True)

seed_master = 1234


def make_panel_process(panel_name: str, phi: float, rng: np.random.Generator):
    if panel_name == "Stationary":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.0,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
        )
    if panel_name == "Trend + shocks":
        return AR1Process(
            mean=0.0,
            phi=phi,
            noise_std=1.0,
            trend_slope=0.001,
            seasonal_amplitude=0.0,
            seasonal_period=None,
            seasonal_phase=0.0,
            rng=rng,
            shock_type="permanent",
            jump_prob=0.005,
            jump_scale=2.0,
        )
    raise ValueError(f"Unknown panel: {panel_name}")


for panel_idx, panel_name in enumerate(panel_names_ablation):
    processes, labels = [], []
    for phi_idx, phi in enumerate(phis):
        rng = np.random.default_rng(seed_master + 1_000 * panel_idx + 10 * phi_idx)
        processes.append(make_panel_process(panel_name, phi, rng))
        labels.append(rf"phi={phi:g}")

    _ = run_sweeps_for_process_list(
        processes,
        labels=labels,
        outdir=abl_dir / panel_name,
        seed=seed_master + panel_idx,
        sample_size=sample_size,
        smoothing_grid=smoothing_grid,
        n_series=n_series,
        burn_in=burn_in,
        alpha=alpha,
        var_warmup=var_warmup,
        base_exp_kwargs=base_exp_kwargs_ablation,
        exp_kwargs_overrides=exp_kwargs_overrides_ablation,
        progress=True,
        parallel=True,
        n_jobs=-1,
    )

print("Done. Ablations saved under:", abl_dir.resolve())